# 01a — Groundsource: Urban Areas Detection

**Purpose:** Intersect each flood polygon with the GHS-BUILT (Global Human Settlements Built-up
Surface) raster to quantify the built-up area contained within it. This produces an `urban_percentage`
column that is used in all downstream steps to filter for genuinely urban flood events.

**Method:** Zonal statistics (sum of GHS-BUILT pixel values inside each polygon) are computed in
Mollweide equal-area projection (ESRI:54009), matching the native CRS of the raster. Processing
is parallelised across multiple CPU cores using `concurrent.futures.ProcessPoolExecutor`.

**Input:**
- `groundsource_2026.parquet` — raw flood events (geometry as WKB bytes, WGS84)
- `GHS_BUILT_S_E2020_GLOBE_R2023A_54009_100_V1_0.tif` — GHS-BUILT raster (100 m, Mollweide)

**Output:**
- `outputs/groundsource_urban_df.parquet` — original columns plus:
  - `urban_built_up_area_m2` : sum of GHS-BUILT pixel values within polygon (m²)
  - `polygon_total_area_m2`  : polygon area in Mollweide projection (m²)
  - `urban_percentage`       : built-up fraction of polygon area (%)

In [1]:
import os
import time
import warnings
from concurrent.futures import ProcessPoolExecutor, as_completed

import numpy as np
import pandas as pd
import geopandas as gpd

warnings.filterwarnings('ignore')

In [2]:
# --- CONFIGURATION ---

INPUT_PATH    = r"D:\Development\RESEARCH\urban_flood_database\Groundsource\groundsource_2026.parquet"
GHS_RASTER    = r"D:\Development\RESEARCH\global_datasets\GHS_BUILT\GHS_BUILT_S_E2020_GLOBE_R2023A_54009_100_V1_0.tif"
OUTPUT_DIR    = r"D:\MY_CODES\global_urban_flood_database\groundsource\outputs"
OUTPUT_PATH   = os.path.join(OUTPUT_DIR, "groundsource_urban_df.parquet")

MOLLWEIDE_CRS = "ESRI:54009"   # native CRS of GHS-BUILT raster; equal-area
WGS84_CRS     = "EPSG:4326"    # CRS of the raw Groundsource geometry
N_WORKERS     = 6              # parallel CPU workers (reduce if RAM is tight)
CHUNK_SIZE    = 5_000          # rows per worker chunk

## 1. Load data and reproject to Mollweide

In [3]:
t_start = time.time()

df = pd.read_parquet(INPUT_PATH)
print(f"Loaded {len(df):,} records in {time.time() - t_start:.1f}s")

# Decode WKB bytes → shapely geometries in WGS84, then reproject to Mollweide
t1 = time.time()
gdf = gpd.GeoDataFrame(df, geometry=gpd.GeoSeries.from_wkb(df['geometry'], crs=WGS84_CRS))
gdf = gdf.to_crs(MOLLWEIDE_CRS)
print(f"Decoded and reprojected {len(gdf):,} geometries in {time.time() - t1:.1f}s")

# Compute polygon area in Mollweide (accurate equal-area measurement)
gdf['polygon_total_area_m2'] = gdf.geometry.area
print(f"Area range: {gdf['polygon_total_area_m2'].min():.0f} – {gdf['polygon_total_area_m2'].max():.0f} m²")

Loaded 2,646,302 records in 2.0s
Decoded and reprojected 2,646,302 geometries in 13.4s
Area range: 1 – 5027064592 m²


## 2. Parallel zonal statistics

In [4]:
def _zonal_stats_worker(args):
    """
    Compute the sum of GHS-BUILT raster pixels within each polygon for one chunk.

    This function runs in a separate process. Geometries are passed as WKB bytes
    (serialisable) and reconstructed inside the worker.

    Args:
        args: tuple of (wkb_bytes_list, raster_path, mollweide_crs)

    Returns:
        list of float — urban built-up area sum in m² for each polygon in the chunk
    """
    wkb_bytes_list, raster_path, mollweide_crs = args
    import geopandas as gpd
    from rasterstats import zonal_stats

    geoms = gpd.GeoSeries.from_wkb(wkb_bytes_list, crs=mollweide_crs)
    results = zonal_stats(
        list(geoms),
        raster_path,
        stats=['sum'],
        all_touched=True,
        nodata=0
    )
    return [r.get('sum', 0) or 0 for r in results]

In [ ]:
# Serialise reprojected geometries to WKB so they can be sent to worker processes
wkb_mollweide = gdf.geometry.to_wkb()

# Split into equal chunks
chunks = [
    wkb_mollweide.iloc[i : i + CHUNK_SIZE].tolist()
    for i in range(0, len(gdf), CHUNK_SIZE)
]
tasks = [(chunk, GHS_RASTER, MOLLWEIDE_CRS) for chunk in chunks]

print(f"Running zonal statistics: {len(chunks)} chunks × {CHUNK_SIZE} rows, {N_WORKERS} workers")

urban_areas_by_chunk = [None] * len(chunks)
t2 = time.time()

with ProcessPoolExecutor(max_workers=N_WORKERS) as executor:
    futures = {executor.submit(_zonal_stats_worker, task): i for i, task in enumerate(tasks)}
    completed = 0
    for future in as_completed(futures):
        i = futures[future]
        urban_areas_by_chunk[i] = future.result()
        completed += 1
        elapsed = time.time() - t2
        pct = completed / len(chunks) * 100
        rate = completed / elapsed * CHUNK_SIZE
        print(f"  {completed}/{len(chunks)} chunks done  ({pct:.1f}%)  "
              f"elapsed {elapsed:.0f}s  ~{rate:.0f} rows/s", end='\r')

print(f"\nZonal statistics complete in {time.time() - t2:.1f}s")

Running zonal statistics: 530 chunks × 5000 rows, 6 workers


## 3. Calculate urban percentage and save

In [ ]:
# Flatten results and attach to GeoDataFrame
all_urban_areas = [val for chunk_result in urban_areas_by_chunk for val in chunk_result]
gdf['urban_built_up_area_m2'] = all_urban_areas

# Urban percentage: built-up area as a fraction of total polygon area
gdf['urban_percentage'] = np.where(
    gdf['polygon_total_area_m2'] > 0,
    (gdf['urban_built_up_area_m2'] / gdf['polygon_total_area_m2']) * 100,
    0.0
)
# Clip to [0, 100] to absorb any floating-point rounding
gdf['urban_percentage'] = gdf['urban_percentage'].clip(0, 100)

print("Urban percentage statistics:")
print(gdf['urban_percentage'].describe())
print(f"\nEvents with urban_percentage > 0 : {(gdf['urban_percentage'] > 0).sum():,}")

In [ ]:
# Build output DataFrame: preserve original WKB geometry column, add new numeric columns
output_df = df.copy()
output_df['polygon_total_area_m2']  = gdf['polygon_total_area_m2'].values
output_df['urban_built_up_area_m2'] = gdf['urban_built_up_area_m2'].values
output_df['urban_percentage']       = gdf['urban_percentage'].values

os.makedirs(OUTPUT_DIR, exist_ok=True)
output_df.to_parquet(OUTPUT_PATH, index=False)

print(f"Saved {len(output_df):,} rows → {OUTPUT_PATH}")
print(f"Total runtime: {time.time() - t_start:.1f}s")